# LSTM-SVR-L model

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('../data/df_with_2regimes.csv')

# take log of realized variance
df['log_realized_variance'] = np.log(df['realized_variance'])
# lag 1,2 andd 3 of realized variance
df['rv_lag1'] = df['log_realized_variance'].shift(1)
df['rv_lag2'] = df['log_realized_variance'].shift(2)
df['rv_lag3'] = df['log_realized_variance'].shift(3)

# rolling mean of realized volatility over the past 30 days
df['rv_rolling_mean_30'] = df['log_realized_variance'].rolling(window=30).mean() 

# take log of realized variance
df['log_realized_variance'] = np.log(df['realized_variance'])

# drop rows with NaN values (due to lag and rolling mean)
df = df.dropna().reset_index(drop=True)
df.head(30)
df.info()
# start from 2017-07-13
df = df[df['date'] >= '2017-07-13'].reset_index(drop=True)
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3489 entries, 0 to 3488
Data columns (total 42 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   date                                        3489 non-null   object 
 1   log_return                                  3489 non-null   float64
 2   realized_variance                           3489 non-null   float64
 3   realized_volatility                         3489 non-null   float64
 4   gtrend_pct_change                           3489 non-null   float64
 5   blockchain_diff_log_n_transactions          3489 non-null   float64
 6   blockchain_diff_log_transaction_fee_usd     3489 non-null   float64
 7   blockchain_diff_log_n_unique_addresses      3489 non-null   float64
 8   blockchain_diff_log_transaction_volume_usd  3489 non-null   float64
 9   log_volume                                  3489 non-null   float64
 10  target      

In [3]:
# start from 2017-07-13
df = df[df['date'] >= '2017-07-13'].reset_index(drop=True)
df.head()
df.tail()

,date,log_return,realized_variance,realized_volatility,gtrend_pct_change,blockchain_diff_log_n_transactions,blockchain_diff_log_transaction_fee_usd,blockchain_diff_log_n_unique_addresses,blockchain_diff_log_transaction_volume_usd,log_volume,...,p_sigma,p_r,p_d,regime,regime_id,log_realized_variance,rv_lag1,rv_lag2,rv_lag3,rv_rolling_mean_30
3120,2026-01-27,0.009776,0.000485,0.022018,-0.061538,-0.104979,-0.046198,0.021370,-0.235868,8.775873,...,0.866667,0.700000,0.583333,High,1.0,-7.631798,-7.940211,-8.366620,-10.489060,-8.413804
3121,2026-01-28,0.000510,0.000319,0.017847,-0.065574,-0.075153,0.240570,0.021306,0.359595,8.800436,...,0.533333,0.533333,0.500000,High,1.0,-8.051862,-7.631798,-7.940211,-8.366620,-8.431558
3122,2026-01-29,-0.053552,0.001190,0.034490,0.333333,0.243422,-0.024924,0.071430,0.118684,9.606559,...,0.966667,0.033333,0.966667,High,1.0,-6.734192,-8.051862,-7.631798,-7.940211,-8.370003
3123,2026-01-30,-0.004770,0.000855,0.029246,0.171053,-0.118552,0.068726,0.107636,0.416170,9.631273,...,0.933333,0.300000,0.816667,High,1.0,-7.064056,-6.734192,-8.051862,-7.631798,-8.314689
3124,2026-01-31,-0.067155,0.000998,0.031598,0.123596,0.062615,-0.051867,-0.096210,-0.546907,9.606164,...,0.933333,0.033333,0.950000,High,1.0,-6.909322,-7.064056,-6.734192,-8.051862,-8.224822


In [4]:
lstm_features = ["log_return", "gtrend_pct_change",
    "blockchain_diff_log_n_transactions",
    "blockchain_diff_log_transaction_fee_usd",
    "blockchain_diff_log_n_unique_addresses",
    "blockchain_diff_log_transaction_volume_usd",
    "log_volume",
    "gold_close_ret",
    "silver_close_ret",
    "brent_close_ret",
    "dji_close_ret",
    "spx_close_ret",
    "rut_close_ret",
    "nasdaq_close_ret",
    "usdcny_close_ret",
    "usdeur_close_ret",
    "gold_volume_chg",
    "silver_volume_chg",
    "brent_volume_chg",
    "dji_volume_chg",
    "spx_volume_chg",
    "rut_volume_chg",
    "nasdaq_volume_chg",
    "vix_close_chg",
    "hash-rate_chg",
    "difficulty_chg",
    "median-confirmation-time_chg",
    "blockchain_log_mempool_count_chg"
    ]

svr_features = ["rv_lag1", "rv_lag2", "rv_lag3", "rv_rolling_mean_30"]

## Weighted LSTM Objective

Each observation is assigned a weight

$$
w_t = (1-\lambda)\frac{T}{\sum_{s=1}^{T} \mathbf{1}\{g_t = g_s\}}
+
\lambda
\frac{\sum_{s=1}^{T} \mathbf{1}\{g_s \ne g_{s-1}\}}
{\sum_{s=1}^{T} \mathbf{1}\{g_t = g_s\}\mathbf{1}\{g_s \ne g_{s-1}\}}
$$

where

- $T$ is the number of observations  
- $g_t$ is the regime label at time $t$  
- $\mathbf{1}\{\cdot\}$ is the indicator function  
- $\lambda \in [0,1]$ controls the emphasis on regime switching observations.

The LSTM is trained using a **weighted categorical cross-entropy loss**

$$
\mathcal{L} =
-\frac{1}{T}
\sum_{t=1}^{T}
w_t
\sum_{k=1}^{K}
y_{t,k}\log(\hat{y}_{t,k})
$$

where

- $K$ = number of regimes  
- $y_{t,k}$ = true indicator of regime $k$ at time $t$  
- $\hat{y}_{t,k}$ = predicted probability from the LSTM softmax layer  
- $w_t$ = observation weight.

In [5]:
def compute_weights(regimes, lam=0.5, eps=1e-12):
    regimes = np.asarray(regimes).astype(int)
    T = len(regimes)

    switch_flag = np.zeros(T, dtype=int)
    switch_flag[1:] = (regimes[1:] != regimes[:-1]).astype(int)

    unique_regimes = np.unique(regimes)

    class_count = {g: np.sum(regimes == g) for g in unique_regimes}
    switch_count = {
        g: np.sum((regimes == g) & (switch_flag == 1))
        for g in unique_regimes
    }
    total_switches = np.sum(switch_flag)

    weights = np.zeros(T, dtype=float)

    for t in range(T):
        g = regimes[t]

        imbalance_term = T / (class_count[g] + eps)

        if total_switches > 0 and switch_count[g] > 0:
            switch_term = total_switches / (switch_count[g] + eps)
        else:
            switch_term = 0.0 

        weights[t] = (1 - lam) * imbalance_term + lam * switch_term 

    # normalize weights for stability
    weights = weights / np.mean(weights)

    return weights, switch_flag

In [6]:
# use this for evaluation and testing
def create_rolling_scaled_sequences_range(
    df, feature_cols, target_col, lookback, scale_window, start_idx, end_idx, date_col=None
):
    X, y, dates = [], [], []

    first_valid_idx = max(start_idx, lookback - 1, scale_window - 1)

    for t in range(first_valid_idx, end_idx - 1):

        hist_window = df[feature_cols].iloc[t - scale_window + 1:t + 1]
        seq_window = df[feature_cols].iloc[t - lookback + 1:t + 1]

        target = df[target_col].iloc[t + 1]

        if hist_window.isnull().any().any():
            continue
        if seq_window.isnull().any().any():
            continue
        if pd.isna(target):
            continue

        scaler = StandardScaler()
        scaler.fit(hist_window)

        seq_scaled = scaler.transform(seq_window)

        X.append(seq_scaled)
        y.append(target)

        if date_col is not None:
            dates.append(df[date_col].iloc[t + 1])
        else:
            dates.append(t + 1)

    return np.array(X), np.array(y), dates

In [7]:
def create_rolling_scaled_sequences_range_weights(
    df, feature_cols, target_col, lookback, scale_window,
    start_idx, end_idx, weights_full, date_col=None
):
    X, y, dates, target_idx, w = [], [], [], [], []

    first_valid_idx = max(start_idx, lookback, scale_window)

    for t in range(first_valid_idx, end_idx - 1):
        hist_window = df[feature_cols].iloc[t - scale_window + 1:t + 1]
        seq_window = df[feature_cols].iloc[t - lookback + 1:t + 1]
        target = df[target_col].iloc[t + 1]

        weight = weights_full[t + 1]

        if hist_window.isnull().any().any():
            continue
        if seq_window.isnull().any().any():
            continue
        if pd.isna(target):
            continue
        if pd.isna(weight):
            continue

        scaler = StandardScaler()
        scaler.fit(hist_window)
        seq_scaled = scaler.transform(seq_window)

        X.append(seq_scaled)
        y.append(target)
        w.append(weight)
        target_idx.append(t + 1)

        if date_col is not None:
            dates.append(df[date_col].iloc[t + 1])
        else:
            dates.append(t + 1)

    return np.array(X), np.array(y), dates, np.array(w), np.array(target_idx)


def scale_rolling(df_hist, feature_cols, lookback, scale_window=730):

    scaler_hist = df_hist[feature_cols].iloc[-min(scale_window, len(df_hist)):].copy()

    scaler = StandardScaler()
    scaler.fit(scaler_hist)

    seq_raw = df_hist[feature_cols].iloc[-lookback:].copy()

    seq_scaled = scaler.transform(seq_raw)

    return seq_scaled, scaler


In [8]:
def build_lstm_model(lookback, n_features, n_regimes=2, units=64, dropout=0.2, optimizer="adam"):
    model = Sequential([
        LSTM(units, input_shape=(lookback, n_features)),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(n_regimes, activation="softmax")
    ])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [9]:
def rolling_regime_forecast(
    df_full,
    lstm_model,
    feature_cols,
    lookback,
    start_idx,
    end_idx,
    scale_window=730,
    date_col="date",
    target_col="regime_id"
):

    results = []

    for t in range(start_idx - 1, end_idx - 1):

        df_hist = df_full.iloc[:t+1].copy()

        if len(df_hist) < lookback:
            continue

        try:
            seq_scaled, _ = scale_rolling(
                df_hist,
                feature_cols=feature_cols,
                lookback=lookback,
                scale_window=scale_window
            )
        except Exception:
            continue

        X_input = np.expand_dims(seq_scaled, axis=0)

        pred_probs = lstm_model.predict(X_input, verbose=0)
        pred_regime = int(np.argmax(pred_probs))

        results.append({
            "date": df_full.iloc[t][date_col],
            "pred_regime": pred_regime,
            "pred_prob_0": pred_probs[0,0],
            "pred_prob_1": pred_probs[0,1]
        })

    return pd.DataFrame(results)

# SVR-Linear

In [10]:
def one_step_svr_forecast(fitted_svr, x_next):
    
    if fitted_svr is None:
        return np.nan

    try:
        x_next = np.asarray(x_next).reshape(1, -1)
        pred = fitted_svr.predict(x_next)[0]
        return float(pred)
    except Exception:
        return np.nan

    
def fit_svr_safe(X_train, y_train, C=1.0, epsilon=1e-4, min_obs=100):
    if len(X_train) < min_obs:
        return None

    try:
        model = LinearSVR(C=C, epsilon=epsilon, dual=True, max_iter=20000)
        model.fit(X_train, y_train)
        return model
    except Exception as e:
        print("LinearSVR fit failed:", e)
        return None

In [11]:
def rolling_lstm_svr_linear_eval(
    df_full,
    regime_forecast_df,
    start_idx,
    end_idx,
    x_feature_cols,
    realized_var_col="log_realized_variance",
    date_col="date",
    rolling_window=365,
    C=1.0,
    epsilon=1e-4,
    min_obs=200,
    use_prob_1_only=True
):
    results = []

    df = df_full.copy()
    regime_df = regime_forecast_df.copy()

    merge_cols = [date_col, "pred_regime", "pred_prob_0", "pred_prob_1"]
    df = df.merge(regime_df[merge_cols], on=date_col, how="left")

    # target is next day's realized variance
    df["rv_lead1"] = df[realized_var_col].shift(-1)

    # row t stores regime forecast for t+1 made using info up to t
    df["prob1_for_next_day"] = df["pred_prob_1"]
    df["prob0_for_next_day"] = df["pred_prob_0"]
    df["regime_for_next_day"] = df["pred_regime"]

    prob_cols = ["prob1_for_next_day"] if use_prob_1_only else ["prob0_for_next_day", "prob1_for_next_day"]
    feature_cols = x_feature_cols + prob_cols

    for t in range(start_idx - 1, end_idx - 1):
        current_row = df.iloc[t]
        next_row = df.iloc[t + 1]

        current_date = current_row[date_col]
        next_date = next_row[date_col]

        # training ends at t-1, so row t is excluded
        window_df = df.iloc[max(0, t - rolling_window + 1): t].copy()

        needed_cols = feature_cols + ["rv_lead1"]
        train_df = window_df[needed_cols].dropna()

        if len(train_df) < min_obs:
            results.append({
                "date": next_date,
                "forecast_origin_date": current_date,
                "actual_var": next_row[realized_var_col],
                "pred_regime": current_row.get("regime_for_next_day", np.nan),
                "pred_prob_0": current_row.get("prob0_for_next_day", np.nan),
                "pred_prob_1": current_row.get("prob1_for_next_day", np.nan),
                "var_svr": np.nan
            })
            continue

        # split continuous features and probability features
        X_train_cont = train_df[x_feature_cols].values
        X_train_prob = train_df[prob_cols].values
        y_train = train_df["rv_lead1"].values

        # scale only continuous features
        scaler = StandardScaler()
        X_train_cont_scaled = scaler.fit_transform(X_train_cont)

        # combine scaled continuous features with raw probabilities
        X_train_final = np.hstack([X_train_cont_scaled, X_train_prob])

        model = fit_svr_safe(
            X_train=X_train_final,
            y_train=y_train,
            C=C,
            epsilon=epsilon,
            min_obs=min_obs
        )

        # prepare current row features for forecasting t+1
        x_t_cont = current_row[x_feature_cols]
        x_t_prob = current_row[prob_cols]

        if x_t_cont.isna().any() or x_t_prob.isna().any():
            pred = np.nan
        else:
            x_t_cont_scaled = scaler.transform(x_t_cont.values.reshape(1, -1))
            x_t_prob_arr = x_t_prob.values.reshape(1, -1)
            x_t_final = np.hstack([x_t_cont_scaled, x_t_prob_arr])

            pred = one_step_svr_forecast(model, x_t_final)

        results.append({
            "date": next_date,
            "forecast_origin_date": current_date,
            "actual_log_var": next_row[realized_var_col],
            "pred_regime": current_row.get("regime_for_next_day", np.nan),
            "pred_prob_0": current_row.get("prob0_for_next_day", np.nan),
            "pred_prob_1": current_row.get("prob1_for_next_day", np.nan),
            "log_var_svr": pred
        })

    return pd.DataFrame(results)

#### metrics

In [12]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))


def qlike(test_actuals, test_preds): 
    test_actuals = np.asarray(test_actuals, dtype=float)
    test_preds = np.asarray(test_preds, dtype=float)

    test_qlike = np.mean(test_actuals / (test_preds + 1e-10) - np.log(test_actuals / (test_preds + 1e-10)) - 1)

    return test_qlike 


In [13]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import backend as K
import tensorflow as tf
import gc
from itertools import product 
from sklearn.svm import SVR
from sklearn.svm import LinearSVR
from sklearn.preprocessing import StandardScaler


In [14]:
n = len(df)

train_size = int(n * 0.60)  # 60/25/15 split
eval_size  = int(n * 0.25)

train_start_idx = 0
train_end_idx   = train_size

eval_start_idx = train_end_idx
eval_end_idx   = eval_start_idx + eval_size

test_start_idx = eval_end_idx
test_end_idx   = n

# split eval into A and B
evalA_size = eval_size // 2

evalA_start_idx = eval_start_idx
evalA_end_idx   = evalA_start_idx + evalA_size

evalB_start_idx = evalA_end_idx
evalB_end_idx   = eval_end_idx

train_df = df.iloc[train_start_idx:train_end_idx].copy()
evalA_df = df.iloc[evalA_start_idx:evalA_end_idx].copy()
evalB_df = df.iloc[evalB_start_idx:evalB_end_idx].copy()
test_df  = df.iloc[test_start_idx:test_end_idx].copy()

print(f"Train set:  {len(train_df)} rows")
print(f"Eval A set: {len(evalA_df)} rows")
print(f"Eval B set: {len(evalB_df)} rows")
print(f"Test set:   {len(test_df)} rows")

Train set:  1875 rows
Eval A set: 390 rows
Eval B set: 391 rows
Test set:   469 rows


In [15]:
train_start_idx = 0
train_end_idx = train_size

eval_start_idx = train_size
eval_end_idx = train_size + eval_size

test_start_idx = eval_end_idx
test_end_idx = len(df)

In [16]:
lstm_grid = {
    "lookback": [20, 30, 60],
    "units": [32, 64],
    "dropout": [0.0, 0.1, 0.2],
    "batch_size": [32,64]
}

lstm_param_grid = []
for lb, units, drop, bs in product(
    lstm_grid["lookback"],
    lstm_grid["units"],
    lstm_grid["dropout"],
    lstm_grid["batch_size"]
):
    lstm_param_grid.append({
        "lookback": lb,
        "units": units,
        "dropout": drop,
        "batch_size": bs
    })


svr_grid = {
    "kernel": ["linear"],
    "C": [0.1, 1.0, 10.0],
    "epsilon": [1e-4, 1e-3, 1e-2]
}

svr_param_grid = []
for kernel, C, epsilon in product(
    svr_grid["kernel"],
    svr_grid["C"],
    svr_grid["epsilon"]
):
    svr_param_grid.append({
        "kernel": kernel,
        "C": C,
        "epsilon": epsilon
    })

print(f"Total LSTM configs: {len(lstm_param_grid)}")
print(f"Total SVR configs: {len(svr_param_grid)}")

Total LSTM configs: 36
Total SVR configs: 9


#### Tune LSTM on fixed SVR

In [21]:
fixed_svr_params = {
    "kernel": "linear",
    "C": 1.0,
    "epsilon": 1e-4
}

stage2_window = 365
min_obs = 200

lstm_results = []

for i, params in enumerate(lstm_param_grid, 1):
    lb = params["lookback"]
    units = params["units"]
    drop = params["dropout"]
    bs = params["batch_size"]

    print(f"\n[{i}/{len(lstm_param_grid)}] Testing LSTM: {params}")

    try:
        K.clear_session()
        tf.keras.backend.clear_session()
        gc.collect()

        # -----------------------------
        # 1) Train LSTM on TRAIN only
        # -----------------------------
        train_weights_full, train_switch_flags = compute_weights(
            train_df["regime_id"].values, lam=0.5
        )

        X_train_lstm, y_train_regime, train_dates, w_train, train_idx = (
            create_rolling_scaled_sequences_range_weights(
                df=train_df,
                feature_cols=lstm_features,
                target_col="regime_id",
                lookback=lb,
                scale_window=365,
                start_idx=0,
                end_idx=len(train_df),
                weights_full=train_weights_full,
                date_col="date"
            )
        )

        w_train = w_train.astype(np.float32)

        # -----------------------------------
        # 2) Validate LSTM on Eval A only
        # -----------------------------------
        X_evalA_lstm, y_evalA_regime, evalA_dates = create_rolling_scaled_sequences_range(
            df=df,
            feature_cols=lstm_features,
            target_col="regime_id",
            lookback=lb,
            scale_window=365,
            start_idx=evalA_start_idx,
            end_idx=evalA_end_idx,
            date_col="date"
        )

        if len(X_train_lstm) == 0 or len(X_evalA_lstm) == 0:
            print("Skipping: no sequences produced.")
            lstm_results.append({
                **params,
                "evalB_rmse": np.nan,
                "evalB_qlike": np.nan,
                "n_evalB_forecasts": 0,
                "error": "no sequences produced"
            })
            continue

        y_train_cat = to_categorical(y_train_regime, num_classes=2)
        y_evalA_cat = to_categorical(y_evalA_regime, num_classes=2)

        model = build_lstm_model(
            lookback=lb,
            n_features=len(lstm_features),
            n_regimes=2,
            units=units,
            dropout=drop,
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
        )

        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train_lstm,
            y_train_cat,
            sample_weight=w_train,
            validation_data=(X_evalA_lstm, y_evalA_cat),
            epochs=100,
            batch_size=bs,
            callbacks=[early_stop],
            verbose=0
        )

        # ---------------------------------------------------------
        # 3) Generate regime probabilities on Eval A + Eval B
        #    starting early enough for stage-2 rolling history
        # ---------------------------------------------------------
        regime_start_idx = max(0, evalA_start_idx - stage2_window)

        evalAB_regime_roll = rolling_regime_forecast(
            df_full=df,
            lstm_model=model,
            feature_cols=lstm_features,
            lookback=lb,
            start_idx=regime_start_idx,
            end_idx=evalB_end_idx,
            scale_window=365,
            date_col="date",
            target_col="regime_id"
        )

        # ------------------------------------------------
        # 4) Tune/evaluate stage-2 SVR on Eval B only
        # ------------------------------------------------
        evalB_stage2 = rolling_lstm_svr_linear_eval(
            df_full=df,
            regime_forecast_df=evalAB_regime_roll,
            start_idx=evalB_start_idx,
            end_idx=evalB_end_idx,
            x_feature_cols=svr_features,
            realized_var_col="log_realized_variance",
            date_col="date",
            rolling_window=stage2_window,
            C=fixed_svr_params["C"],
            epsilon=fixed_svr_params["epsilon"],
            min_obs=min_obs,
            use_prob_1_only=True
        )

        evalB_rmse = rmse(evalB_stage2["actual_log_var"], evalB_stage2["log_var_svr"])
        # convert to variance scale for qlike
        evalB_qlike = qlike(np.exp(evalB_stage2["actual_log_var"]), np.exp(evalB_stage2["log_var_svr"]))
        n_evalB_forecasts = evalB_stage2["log_var_svr"].notna().sum()

        lstm_results.append({
            **params,
            "evalB_rmse": evalB_rmse,
            "evalB_qlike": evalB_qlike,
            "n_evalB_forecasts": n_evalB_forecasts
        })

    except Exception as e:
        print(f"Error for params {params}: {e}")
        lstm_results.append({
            **params,
            "evalB_rmse": np.nan,
            "evalB_qlike": np.nan,
            "n_evalB_forecasts": 0,
            "error": str(e)
        })

lstm_results_df = pd.DataFrame(lstm_results).sort_values("evalB_qlike")
print(lstm_results_df.head())


[1/36] Testing LSTM: {'lookback': 20, 'units': 32, 'dropout': 0.0, 'batch_size': 32}


In [20]:
lstm_results

[{'lookback': 20,
  'units': 32,
  'dropout': 0.0,
  'batch_size': 32,
  'evalB_rmse': 0.9819884388367806,
  'evalB_qlike': 0.5852605700946022,
  'n_evalB_forecasts': 391}]

#### Tune svr on best LSTM parameters

In [ ]:
lstm_results_df.head()

In [ ]:
best_lstm_params = lstm_results_df.iloc[0].to_dict()
print(best_lstm_params)

In [ ]:
K.clear_session()
tf.keras.backend.clear_session()
gc.collect()

best_lb = int(best_lstm_params["lookback"])
best_units = int(best_lstm_params["units"])
best_drop = float(best_lstm_params["dropout"])
best_bs = int(best_lstm_params["batch_size"])

train_weights_full, train_switch_flags = compute_weights(
    train_df["regime_id"].values, lam=0.5
)

# 1) Train LSTM on TRAIN only
X_train_lstm, y_train_regime, train_dates, w_train, train_idx = (
    create_rolling_scaled_sequences_range_weights(
        df=train_df,
        feature_cols=lstm_features,
        target_col="regime_id",
        lookback=best_lb,
        scale_window=365,
        start_idx=0,
        end_idx=len(train_df),
        weights_full=train_weights_full,
        date_col="date"
    )
)

w_train = w_train.astype(np.float32)
y_train_cat = to_categorical(y_train_regime, num_classes=2)

# 2) Validate / early stop on EVAL A only
X_evalA_lstm, y_evalA_regime, evalA_dates = create_rolling_scaled_sequences_range(
    df=df,
    feature_cols=lstm_features,
    target_col="regime_id",
    lookback=best_lb,
    scale_window=365,
    start_idx=evalA_start_idx,
    end_idx=evalA_end_idx,
    date_col="date"
)

y_evalA_cat = to_categorical(y_evalA_regime, num_classes=2)

best_lstm_model = build_lstm_model(
    lookback=best_lb,
    n_features=len(lstm_features),
    n_regimes=2,
    units=best_units,
    dropout=best_drop,
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

best_lstm_model.fit(
    X_train_lstm,
    y_train_cat,
    sample_weight=w_train,
    validation_data=(X_evalA_lstm, y_evalA_cat),
    epochs=100,
    batch_size=best_bs,
    callbacks=[early_stop],
    verbose=0
)

# 3) Generate fixed regime probabilities for SVR tuning on Eval B
# start early enough so stage-2 has rolling history
stage2_window = 365
regime_start_idx = max(0, evalB_start_idx - stage2_window)

eval_regime_fixed = rolling_regime_forecast(
    df_full=df,
    lstm_model=best_lstm_model,
    feature_cols=lstm_features,
    lookback=best_lb,
    start_idx=regime_start_idx,
    end_idx=evalB_end_idx,
    scale_window=365,
    date_col="date",
    target_col="regime_id"
)

In [ ]:
svr_results = []

for i, params in enumerate(svr_param_grid, 1):
    kernel = params["kernel"]
    C = params["C"]
    epsilon = params["epsilon"]

    print(f"\n[{i}/{len(svr_param_grid)}] Testing SVR: {params}")

    try:
        evalB_svr = rolling_lstm_svr_linear_eval(
            df_full=df,
            regime_forecast_df=eval_regime_fixed,
            start_idx=evalB_start_idx,
            end_idx=evalB_end_idx,
            x_feature_cols=svr_features,
            realized_var_col="log_realized_variance",
            date_col="date",
            rolling_window=365,
            C=C,
            epsilon=epsilon,
            min_obs=200,
            use_prob_1_only=True,
            kernel=kernel
        )

        evalB_rmse = rmse(evalB_stage2["actual_log_var"], evalB_stage2["log_var_svr"])
        # convert to variance scale for qlike
        evalB_qlike = qlike(np.exp(evalB_stage2["actual_log_var"]), np.exp(evalB_stage2["log_var_svr"]))
        n_evalB_forecasts = evalB_stage2["log_var_svr"].notna().sum()

        svr_results.append({
            **params,
            "evalB_rmse": evalB_rmse,
            "evalB_qlike": evalB_qlike,
            "n_evalB_forecasts": n_evalB_forecasts
        })

    except Exception as e:
        svr_results.append({
            **params,
            "evalB_rmse": np.nan,
            "evalB_qlike": np.nan,
            "n_evalB_forecasts": 0,
            "error": str(e)
        })

svr_results_df = pd.DataFrame(svr_results).sort_values("evalB_qlike")
print(svr_results_df.head())

In [ ]:
best_svr_params = svr_results_df.iloc[0].to_dict()
print(best_svr_params)

### out of sample testing

In [ ]:
test_regime_start_idx = max(0, test_start_idx - 365)

test_regime_roll = rolling_regime_forecast(
    df_full=df,
    lstm_model=best_lstm_model,
    feature_cols=lstm_features,
    lookback=best_lb,
    start_idx=test_regime_start_idx,
    end_idx=test_end_idx,
    scale_window=365,
    date_col="date",
    target_col="regime_id"
)

test_svr = rolling_lstm_svr_linear_eval(
    df_full=df,
    regime_forecast_df=test_regime_roll,
    start_idx=test_start_idx,
    end_idx=test_end_idx,
    x_feature_cols=svr_features,
    realized_var_col="log_realized_variance",
    date_col="date",
    rolling_window=365,
    C=float(best_svr_params["C"]),
    epsilon=float(best_svr_params["epsilon"]),
    min_obs=200,
    use_prob_1_only=True
)

# RMSE on log-variance scale
test_rmse = rmse(
    test_svr["actual_log_var"],
    test_svr["log_var_svr"]
)

# RSME on variance scale 
test_rmse_var = rmse(
    np.exp(test_svr["actual_log_var"]),
    test_svr["log_var_svr"]
)

# QLIKE on variance scale
test_qlike = qlike(
    np.exp(test_svr["actual_log_var"]),
    test_svr["log_var_svr"]
)

print("Final Test RMSE :", test_rmse)
print("Final Test RMSE (var scale):", test_rmse_var)
print("Final Test QLIKE:", test_qlike)
print("Number of test forecasts:", test_svr["var_soft"].notna().sum())

In [ ]:
test_svr.isna().sum()




In [ ]:
test_svr.tail()

In [ ]:
test_svr.head()

In [ ]:
# plot actual vs predicted variance on test set
import matplotlib.pyplot as plt
test_svr['date'] = pd.to_datetime(test_svr['date'])
plt.figure(figsize=(12,6))
plt.plot(test_svr["date"], test_svr["actual_var"], label="Realized variance")
plt.plot(test_svr["date"], test_svr["var_svr"], label="Predicted variance")
plt.title("Actual vs Predicted Realized Variance on Test Set")
plt.xlabel("Date")
plt.ylabel("Realized Variance")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# plot errors over time
test_svr['date'] = pd.to_datetime(test_svr['date'])
test_svr['error'] = test_svr["actual_var"] - test_svr["var_svr"]
plt.figure(figsize=(12,6))
plt.plot(test_svr["date"], test_svr["error"], label="Forecast Error") 
plt.axhline(0, color='red', linestyle='--', label="Zero Error Line")
plt.title("Forecast Error (Actual - Predicted) Over Time")
plt.xlabel("Date")
plt.ylabel("Forecast Error")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


test_svr['date'] = pd.to_datetime(test_svr['date'])
plt.figure(figsize=(12,6))
plt.plot(test_svr["date"], np.exp(test_svr["actual_var"]), label="Realized variance")
plt.plot(test_svr["date"], np.exp(test_svr["var_svr"]), label="Predicted variance")
plt.title("Actual vs Predicted Realized Variance on Test Set")
plt.xlabel("Date")
plt.ylabel("Realized Variance")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# save final results and export test predictions
test_svr.to_csv("results/lstm-svr1.csv", index=False)


In [ ]:
df['date'] = pd.to_datetime(df['date'])
test_svr['date'] = pd.to_datetime(test_svr['date'])

result_df = test_svr.merge(
    df[["date", "regime_id"]],
    on="date",
    how="left"
)

result_df = result_df.rename(columns={"regime_id": "actual_regime"})

result_df.head(20)

In [ ]:
# classification report of predicted regime vs actual regime on test set
from sklearn.metrics import classification_report
# drop rows with NaN in pred_regime or actual_regime
result_df_clf = result_df.dropna(subset=["pred_regime", "actual_regime"])
print(classification_report(result_df_clf["actual_regime"], result_df_clf["pred_regime"]))
